# 01 — Data Preprocessing

**Goal:** load the medical dataset, explore it, handle missing values, encode labels,
standardize features, split into train/test, and save the processed arrays so that
every downstream notebook (baseline classifier + GA/PSO/GWO/WOA feature selection)
works from an identical, frozen dataset.

This notebook is deliberately self-contained — it does not depend on any other
notebook having run first.

In [ ]:
# If running on Kaggle, uncomment to install any missing packages
# !pip install -q scikit-learn pandas numpy

import sys
sys.path.append("..")  # so we can import the shared utils/ package

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from utils.preprocessing import (
    load_dataset,
    explore_dataset,
    handle_missing_values,
    encode_labels,
    preprocess_and_split,
    save_processed_data,
)

## 1. Load dataset

Default: the Wisconsin Breast Cancer dataset (binary classification, 30 numeric features). Swap `source="csv"` to point at any other medical dataset added to your Kaggle notebook's input directory.

In [ ]:
X, y = load_dataset(source="breast_cancer")
print("Features shape:", X.shape)
print("Target shape:", y.shape)
X.head()

## 2. Explore dataset

In [ ]:
explore_dataset(X, y)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5, 4))
sns.countplot(x=y)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.show()

## 3. Handle missing values

The breast cancer dataset has no missing values, but this step is included so the pipeline generalizes to any real-world medical dataset you swap in.

In [ ]:
X_clean = handle_missing_values(X, strategy="median")
print("Missing values remaining:", X_clean.isnull().sum().sum())

## 4. Encode labels

In [ ]:
y_encoded, label_encoder = encode_labels(y)
print("Classes:", label_encoder.classes_)
print("Encoded label sample:", y_encoded[:10])

## 5. Train/test split

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_clean.values,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded,
)
print("Train shape:", X_train_raw.shape)
print("Test shape:", X_test_raw.shape)

## 6. Normalize / standardize features

We fit the scaler on the training set only, then apply it to the test set, to avoid data leakage.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("X_train mean ~0, std ~1:")
print(X_train.mean(axis=0)[:5].round(3))
print(X_train.std(axis=0)[:5].round(3))

## 7. Save processed datasets

Saved to `../datasets/` so every subsequent notebook (`02_Baseline_Classifier`,
`03_GA`, `04_PSO`, `05_GWO`, `06_WOA`) can load the *exact same* split.

In [ ]:
feature_names = list(X_clean.columns)

save_processed_data(
    X_train, X_test, y_train, y_test,
    feature_names=feature_names,
    out_dir="../datasets",
)

## Summary

| Item | Value |
|---|---|
| Total features | `X_train.shape[1]` |
| Train samples | `X_train.shape[0]` |
| Test samples | `X_test.shape[0]` |

Processed data is saved and ready for `02_Baseline_Classifier.ipynb`.